# 13F and N-PORT data exploration

This notebook is the first point-in-time data-spine review for the asset-embeddings project. Its purpose is to understand the two holdings products before choosing cleaning, aggregation, or modeling rules.

The central question is whether each product can support a defensible investor × asset matrix $H_{i,a,t}$. Economic dates, filing dates, and availability timestamps remain separate throughout.

## Public-repository guardrails

- Private cloud URIs are supplied through environment variables.
- Only bounded samples are downloaded, into the ignored `data/` directory.
- Raw rows and executed notebook outputs are not committed.
- Filters are displayed as audit stages; missing, zero, negative, and excluded observations are not silently merged.
- Results from this notebook are diagnostics, not a production data pipeline or a strategy backtest.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.holdings_exploration import (
    DATASET_SPECS,
    category_profile,
    duplicate_profile,
    fetch_private_object,
    filing_conflict_profile,
    identifier_profile,
    matrix_readiness,
    portfolio_profile,
    read_parquet_sample,
    schema_profile,
    timing_profile,
    value_profile,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

## 1. Configure bounded private inputs

Set the four URI variables shown in `.env.example` in your terminal before launching Jupyter. Keep `FETCH_PRIVATE_OBJECTS = False` unless you intentionally want this notebook to download the configured objects. Exact object access is sufficient; bucket-wide listing is not required.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'exploration'
DATA_DIR.mkdir(parents=True, exist_ok=True)
GCLOUD = PROJECT_ROOT / 'scripts' / 'gcloud'

FILES = {
    '13f': DATA_DIR / '13f_holdings_sample.parquet',
    'nport': DATA_DIR / 'nport_holdings_sample.parquet',
}
DICTIONARIES = {
    '13f': DATA_DIR / '13f_variable_dictionary.parquet',
    'nport': DATA_DIR / 'nport_variable_dictionary.parquet',
}
URI_ENV = {
    FILES['13f']: 'ASSET_EMBEDDINGS_13F_SAMPLE_URI',
    FILES['nport']: 'ASSET_EMBEDDINGS_NPORT_SAMPLE_URI',
    DICTIONARIES['13f']: 'ASSET_EMBEDDINGS_13F_DICTIONARY_URI',
    DICTIONARIES['nport']: 'ASSET_EMBEDDINGS_NPORT_DICTIONARY_URI',
}

FETCH_PRIVATE_OBJECTS = False
if FETCH_PRIVATE_OBJECTS:
    for destination, variable in URI_ENV.items():
        if destination.exists():
            continue
        uri = os.environ.get(variable)
        if not uri:
            raise RuntimeError(f'Missing required environment variable: {variable}')
        fetch_private_object(uri, destination, gcloud_executable=GCLOUD)

pd.DataFrame(
    [{'input': key, 'local_file': str(path.relative_to(PROJECT_ROOT)), 'exists': path.exists()}
     for key, path in {**FILES, **{f'{k}_dictionary': v for k, v in DICTIONARIES.items()}}.items()]
)

In [ ]:
missing_samples = [str(path) for path in FILES.values() if not path.is_file()]
if missing_samples:
    raise FileNotFoundError(
        'Bounded sample files are missing. Configure the private URI environment variables, '
        'set FETCH_PRIVATE_OBJECTS=True, and rerun the configuration cell. Missing: '
        + ', '.join(missing_samples)
    )

samples = {name: read_parquet_sample(path) for name, path in FILES.items()}
pd.DataFrame(
    [{'dataset': DATASET_SPECS[name].name, 'sample_rows': len(frame), 'columns': len(frame.columns)}
     for name, frame in samples.items()]
)

## 2. Observation grain and schema

We first inspect the schema without imposing an equity universe. The expected investor units differ: 13F uses the filing manager, while N-PORT uses the fund series.

In [ ]:
schema_tables = []
for name, frame in samples.items():
    profile = schema_profile(frame)
    profile.insert(0, 'dataset', DATASET_SPECS[name].name)
    schema_tables.append(profile)
schema = pd.concat(schema_tables, ignore_index=True)
display(schema)

In [ ]:
critical_dictionary_fields = {
    '13f': ['manager_cik', 'accession_no', 'period_of_report', 'filed_at',
            'availability_timestamp_utc', 'cusip', 'resolved_issuer_cik',
            'value_usd', 'shares_or_principal_amount', 'put_call'],
    'nport': ['series_id', 'accession_no', 'period_of_report_end', 'filed_at',
              'availability_timestamp_utc', 'cusip', 'isin', 'value_usd',
              'pct_value', 'asset_category', 'payoff_profile'],
}
for name, path in DICTIONARIES.items():
    if not path.is_file():
        print(f'{DATASET_SPECS[name].name}: variable dictionary not downloaded; continuing with Parquet schema.')
        continue
    dictionary = pd.read_parquet(path)
    selected = dictionary[dictionary['column_name'].isin(critical_dictionary_fields[name])].copy()
    display(selected[['table_name', 'column_name', 'dtype', 'role', 'unit',
                      'date_semantics', 'null_policy', 'leakage_risk']])

## 3. Point-in-time timing

The portfolio report date is an economic timestamp. The filing and availability timestamps determine when the observation can enter research or a strategy. The final rows count both observations available before filing and economic dates later than filing; either violation blocks production use until explained.

In [ ]:
for name, frame in samples.items():
    print(DATASET_SPECS[name].name)
    display(timing_profile(frame, DATASET_SPECS[name]))

## 4. Identifier coverage and duplicate keys

Ticker is descriptive, not a timeless asset key. This section measures available identifiers and tests both the product holding ID and the filing-plus-ordinal natural key.

In [ ]:
for name, frame in samples.items():
    spec = DATASET_SPECS[name]
    print(spec.name)
    display(identifier_profile(frame, spec))
    display(duplicate_profile(frame, spec))
    conflicts = filing_conflict_profile(frame, spec)
    print(f'investor-report groups with multiple filings: {len(conflicts)}')
    display(conflicts.head(20))

## 5. Holding values and dataset-specific states

Negative N-PORT values, option indicators, currency, units, and asset categories may be economically meaningful. We profile them before defining any long-equity subset.

In [ ]:
for name, frame in samples.items():
    print(DATASET_SPECS[name].name)
    display(value_profile(frame, DATASET_SPECS[name]))

display(category_profile(
    samples['13f'],
    ['form_type', 'put_call', 'shares_or_principal_type', 'issuer_identity_confidence'],
))
display(category_profile(
    samples['nport'],
    ['asset_category', 'payoff_profile', 'units', 'currency', 'fair_value_level'],
))

## 6. Portfolio-level shape

A portfolio is kept at filing level here. Amendments are not combined with original filings, and manager-level 13F portfolios are not treated as fund-level portfolios.

In [ ]:
for name, frame in samples.items():
    print(DATASET_SPECS[name].name)
    display(portfolio_profile(frame, DATASET_SPECS[name]))

## 7. Provisional embedding readiness

These are diagnostic matrices, not final production universes. Both use CUSIP only, so missing identifiers remain excluded rather than silently replaced by ticker. The function refuses to combine multiple filings for the same investor-report date; amendments must be resolved first. The 13F diagnostic keeps positive reported values; options are still visible in the earlier category table. The N-PORT diagnostic additionally selects the explicitly configured equity-category code. The paper's 20-assets/20-investors rule is enforced iteratively by report date.

In [ ]:
summary_13f, audit_13f, filter_log_13f = matrix_readiness(
    samples['13f'],
    DATASET_SPECS['13f'],
    asset_column='cusip',
    positive_values_only=True,
    minimum_assets_per_investor=20,
    minimum_investors_per_asset=20,
)

summary_nport, audit_nport, filter_log_nport = matrix_readiness(
    samples['nport'],
    DATASET_SPECS['nport'],
    asset_column='cusip',
    positive_values_only=True,
    category_filter=('asset_category', ('EC',)),
    minimum_assets_per_investor=20,
    minimum_investors_per_asset=20,
)

print('13F filter audit')
display(audit_13f)
display(summary_13f)
display(filter_log_13f)

print('N-PORT filter audit')
display(audit_nport)
display(summary_nport)
display(filter_log_nport)

## 8. Decisions required before the data spine

Record conclusions only after reviewing multiple periods, including amendments and stressed periods.

1. **Investor unit:** 13F manager versus N-PORT series/fund.
2. **Asset unit:** security, issuer, or company; mapping must be effective-dated.
3. **Amendments:** replacement, restatement, or as-filed version history.
4. **Availability:** exact permitted timestamp and decision-time buffer.
5. **Position states:** long equity, options, negative exposure, derivatives, cash, and missing coverage.
6. **Units and reconciliation:** value, balance, shares/principal, currency, and portfolio totals.
7. **Cross-product linkage:** whether 13F and N-PORT overlap can be measured without double-counting.
8. **Universe sensitivity:** 20/20 filters, concentration cap, microcap rules, and alternative identifier coverage.

The next milestone is a written data-spine decision record plus tests for amendments, effective-dated identifiers, timestamps, and portfolio reconciliation.